# 🏦 Incremental Campaign Uplift Modeling — Interactive Pipeline

**Treatment arms**: Control, \$100, \$400, \$500  
**`remail` and `stipulation`** are **predictors / covariates** (not treatment arms).  

## Workflow
| Step | Description |
|------|-------------|
| 0 | Data preparation (generate or load) |
| 1 | Feature selection — Step 1: Variance + Correlation pruning |
| 2 | Feature selection — Step 2: Boruta-SHAP |
| 2.5 | Bias correction (PSM / IPTW / none) |
| 3 | Model 1: X-Learner uplift (3 offer arms vs. control) |
| 4 | Model 2: Attrition / retention prediction |
| 4b | Combined insights |
| 5 | Model 3: Net-value optimisation (baseline) |
| 5b | Decile targeting strategy |
| 6 | Scenario analysis (remail × stipulation toggles) |
| 7 | Business recommendations |
| 8 | Save model package |

> **Tip**: Run cells top-to-bottom for the full pipeline, or re-run individual sections after adjusting the configuration cell below.

---
## 📦 Setup & Imports

In [ ]:
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

# ── Make sure the project root is on the path ────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

from sklearn.model_selection import train_test_split

# ── Project modules ──────────────────────────────────────────────────────────
import config
from src.data_generation import generate_epsilon_data, save_data
from src.feature_selection.step1_initial_pruning import run_initial_pruning
from src.feature_selection.step2_boruta_shap import run_boruta_shap
from src.models.xlearner_uplift import train_xlearner
from src.models.attrition_model import train_attrition_model
from src.models.net_value_strategy import optimize_offers, NetValueOptimizer
from src.models.propensity_matching import run_propensity_matching
from src.models.iptw import run_iptw
from src.models.model_registry import save_pipeline

print("✅ All imports successful")

---
## ⚙️ Configuration

Override any `config.py` setting here **without editing the file**.  
Set `OVERRIDE_CONFIG = True` and adjust the values below.

In [ ]:
# ── Optional overrides ────────────────────────────────────────────────────────
OVERRIDE_CONFIG = False   # Set True to apply overrides below

if OVERRIDE_CONFIG:
    config.N_SAMPLES              = 20_000    # Reduce for faster tests, e.g. 5_000
    config.BORUTA_N_TRIALS        = 20        # Reduce for faster tests
    config.BIAS_CORRECTION_METHOD = 'psm'    # 'psm' | 'iptw' | 'none'
    config.USE_MATCHED_DATA_FOR_XLEARNER = False
    print("⚠️  Config overrides applied")

# ── Display current settings ──────────────────────────────────────────────────
settings = {
    'N_SAMPLES':                  config.N_SAMPLES,
    'N_FEATURES':                 config.N_FEATURES,
    'RANDOM_SEED':                config.RANDOM_SEED,
    'BIAS_CORRECTION_METHOD':     config.BIAS_CORRECTION_METHOD,
    'USE_MATCHED_DATA_FOR_XLEARNER': getattr(config, 'USE_MATCHED_DATA_FOR_XLEARNER', False),
    'BORUTA_N_TRIALS':            config.BORUTA_N_TRIALS,
    'VARIANCE_THRESHOLD':         config.VARIANCE_THRESHOLD,
    'CORRELATION_THRESHOLD':      config.CORRELATION_THRESHOLD,
    'OFFER_COST_RATE':            config.OFFER_COST_RATE,
    'REMAIL_COST':                config.REMAIL_COST,
    'STIPULATION_COST':           config.STIPULATION_COST,
    'LOG_TRANSFORM_TARGET':       config.LOG_TRANSFORM_TARGET,
    'TEST_SIZE':                  config.TEST_SIZE,
    'RESULTS_DIR':                config.RESULTS_DIR,
    'MODELS_DIR':                 config.MODELS_DIR,
}

pd.DataFrame.from_dict(settings, orient='index', columns=['Value'])

---
## Step 0 — Data Preparation

In [ ]:
t_pipeline = time.time()
t0 = time.time()

data_path = os.path.join(config.DATA_DIR, 'epsilon_synthetic.csv')

if os.path.exists(data_path):
    print(f"Loading existing data from {data_path} ...")
    df = pd.read_csv(data_path)
    print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")

    if df['treatment'].max() > 3:
        print("⚠️  Old 13-arm design detected — regenerating ...")
        df = generate_epsilon_data()
        save_data(df)
    elif not {'stipulation', 'remail'}.issubset(df.columns):
        print("⚠️  Missing remail/stipulation columns — regenerating ...")
        df = generate_epsilon_data()
        save_data(df)
else:
    print("Generating synthetic Epsilon-like data ...")
    df = generate_epsilon_data()
    save_data(df)

# ── Column split ──────────────────────────────────────────────────────────────
outcome_cols = ['treatment', 'treatment_name', 'opening_balance', 'on_book_month9']
exclude_cols = outcome_cols + ['offer']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X           = df[feature_cols]
y_balance   = df['opening_balance']
y_attrition = df['on_book_month9']
treatment   = df['treatment']
offers_col  = df['offer']
stip_col    = df['stipulation']
remail_col  = df['remail']

print(f"\nData summary:")
print(f"  Total samples   : {X.shape[0]:,}")
print(f"  Feature columns : {X.shape[1]}")
print(f"  remail in features       : {'remail' in X.columns}")
print(f"  stipulation in features  : {'stipulation' in X.columns}")
print(f"  Opening balance : mean=${y_balance.mean():.2f}, std=${y_balance.std():.2f}")
print(f"  Month-9 retention: {y_attrition.mean():.1%}")
print(f"\nStep 0 done in {time.time()-t0:.1f}s")

In [ ]:
# ── Treatment distribution ────────────────────────────────────────────────────
print(f"Treatment distribution ({len(config.TREATMENT_COMPONENTS)} arms):")
rows = []
for arm_id in sorted(config.TREATMENT_COMPONENTS):
    cnt   = (treatment == arm_id).sum()
    pct   = 100 * cnt / len(treatment)
    offer = config.TREATMENT_COMPONENTS[arm_id]
    rows.append({'Arm': arm_id, 'Label': config.TREATMENTS[arm_id],
                 'Offer ($)': offer, 'Count': cnt, 'Pct (%)': round(pct, 1)})
display(pd.DataFrame(rows).set_index('Arm'))

# ── Remail / Stipulation summary ──────────────────────────────────────────────
treated_mask = offers_col > 0
print(f"\nRemail / Stipulation (treated prospects only):")
print(f"  Remail=1      : {remail_col[treated_mask].sum():,}  "
      f"({100*remail_col[treated_mask].mean():.1f}%)")
print(f"  Stipulation=1 : {stip_col[treated_mask].sum():,}  "
      f"({100*stip_col[treated_mask].mean():.1f}%)")

In [ ]:
# ── Quick exploratory plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Opening balance by treatment
for arm_id in sorted(config.TREATMENT_COMPONENTS):
    vals = y_balance[treatment == arm_id]
    axes[0].hist(vals, bins=60, alpha=0.5, label=config.TREATMENTS[arm_id])
axes[0].set_title('Opening Balance Distribution by Treatment Arm')
axes[0].set_xlabel('Opening Balance ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Retention by treatment
retention_by_arm = df.groupby('treatment_name')['on_book_month9'].mean().sort_values()
axes[1].barh(retention_by_arm.index, retention_by_arm.values * 100)
axes[1].set_title('Month-9 Retention Rate by Treatment Arm')
axes[1].set_xlabel('Retention Rate (%)')
axes[1].set_xlim(0, 100)
for i, v in enumerate(retention_by_arm.values):
    axes[1].text(v * 100 + 0.5, i, f'{v:.1%}', va='center')

plt.tight_layout()
plt.show()

---
## Step 1 — Feature Selection: Initial Pruning (Variance + Correlation)

In [ ]:
t1 = time.time()
print(f"Starting Step 1: Variance threshold={config.VARIANCE_THRESHOLD}, "
      f"Correlation threshold={config.CORRELATION_THRESHOLD}")
print(f"Input features: {X.shape[1]}")

step1_report_path = os.path.join(config.RESULTS_DIR, 'step1_pruning_report.txt')
X_step1, pruner = run_initial_pruning(
    X,
    y=y_balance,
    treatment=treatment,
    save_report_path=step1_report_path,
)

print(f"\n✅ Step 1 complete: {X.shape[1]} → {X_step1.shape[1]} features  "
      f"(removed {X.shape[1]-X_step1.shape[1]}, {time.time()-t1:.1f}s)")

In [ ]:
# View the pruning report
with open(step1_report_path) as f:
    print(f.read()[:3000])  # Show first 3000 chars

---
## Step 2 — Feature Selection: Boruta-SHAP

> ⏱️ **Expected runtime**: 5–20 minutes depending on `BORUTA_N_TRIALS` and hardware.  
> Reduce `config.BORUTA_N_TRIALS` (e.g. to `20`) in the Configuration cell above for faster testing.

In [ ]:
t2 = time.time()
print(f"Starting Step 2: Boruta-SHAP with {config.BORUTA_N_TRIALS} trials")
print(f"Input features: {X_step1.shape[1]}")

step2_report_path = os.path.join(config.RESULTS_DIR, 'step2_boruta_report.txt')
X_step2, boruta = run_boruta_shap(
    X_step1,
    y=y_balance,
    task='regression',
    save_report_path=step2_report_path,
)

print(f"\n✅ Step 2 complete: {X_step1.shape[1]} → {X_step2.shape[1]} features  "
      f"(removed {X_step1.shape[1]-X_step2.shape[1]}, {time.time()-t2:.1f}s)")

for col in ['remail', 'stipulation']:
    survived = col in X_step2.columns
    print(f"  {col:>12} survived Boruta: {survived}")

In [ ]:
# ── Sieve summary table ───────────────────────────────────────────────────────
sieve_summary = pd.DataFrame([
    {'Step': 'Original',           'Features': X.shape[1],      'Reduction (%)': 0},
    {'Step': 'After Step 1 (Prune)', 'Features': X_step1.shape[1],
     'Reduction (%)': round(100*(1 - X_step1.shape[1]/X.shape[1]), 1)},
    {'Step': 'After Step 2 (Boruta)', 'Features': X_step2.shape[1],
     'Reduction (%)': round(100*(1 - X_step2.shape[1]/X.shape[1]), 1)},
    {'Step': 'Step 3: L1 in XGBoost', 'Features': '(embedded)',  'Reduction (%)': 'embedded'},
]).set_index('Step')
display(sieve_summary)

# Save final selected features list
selected_features_path = os.path.join(config.RESULTS_DIR, 'selected_features.txt')
with open(selected_features_path, 'w') as f:
    f.write(f"Selected Features ({len(X_step2.columns)}):\n")
    f.write('='*60 + '\n\n')
    for feat in sorted(X_step2.columns):
        f.write(f'  {feat}\n')
print(f"Selected features saved to: {selected_features_path}")

In [ ]:
# View the Boruta report
with open(step2_report_path) as f:
    print(f.read()[:3000])

---
## Step 2.5 — Bias Correction

Method is controlled by `config.BIAS_CORRECTION_METHOD`: `'psm'` | `'iptw'` | `'none'`

In [ ]:
bias_method = getattr(config, 'BIAS_CORRECTION_METHOD', 'psm').lower()
print(f"Bias correction method: {bias_method.upper()}")

t25 = time.time()

# Initialise defaults
X_for_xlearner   = X_step2
t_for_xlearner   = treatment
y_for_xlearner   = y_balance
sample_weight_xl = None

# ── PSM ───────────────────────────────────────────────────────────────────────
if bias_method == 'psm':
    print("\nRunning PSM diagnostics ...")
    psm = run_propensity_matching(
        X=X_step2, treatment=treatment, save_results_dir=config.RESULTS_DIR)

    use_matched = getattr(config, 'USE_MATCHED_DATA_FOR_XLEARNER', False)

    if use_matched:
        X_for_xlearner = pd.DataFrame()
        t_for_xlearner = pd.Series(dtype=int)
        for arm_id, match_df in psm.matched_data.items():
            feat_cols   = [c for c in match_df.columns if c != 'matched_binary_treatment']
            binary_t    = match_df['matched_binary_treatment'].values
            arm_ids_row = np.where(binary_t == 1, arm_id, 0)
            X_for_xlearner = pd.concat([X_for_xlearner, match_df[feat_cols]], ignore_index=True)
            t_for_xlearner = pd.concat([t_for_xlearner, pd.Series(arm_ids_row)], ignore_index=True)
        y_for_xlearner = y_balance.values[:len(X_for_xlearner)]
        print(f"Matched dataset: {len(X_for_xlearner):,} rows ({len(X_for_xlearner)/len(X_step2)*100:.1f}%)")
    else:
        print("PSM for diagnostics only — X-Learner uses full dataset.")

# ── IPTW ──────────────────────────────────────────────────────────────────────
elif bias_method == 'iptw':
    print("\nRunning IPTW ...")
    iptw_result = run_iptw(
        X=X_step2, treatment=treatment, save_results_dir=config.RESULTS_DIR)
    sample_weight_xl = iptw_result.weights
    print(f"Weights computed — min={sample_weight_xl.min():.3f}, "
          f"max={sample_weight_xl.max():.3f}, mean={sample_weight_xl.mean():.3f}")

# ── None ──────────────────────────────────────────────────────────────────────
elif bias_method == 'none':
    print("No bias correction applied — X-Learner uses full unweighted dataset.")
else:
    raise ValueError(f"Unknown BIAS_CORRECTION_METHOD='{bias_method}'. Valid: 'psm', 'iptw', 'none'")

print(f"\n✅ Bias correction done  [{time.time()-t25:.1f}s]")

In [ ]:
# ── Show Love plots / weight diagnostics inline ───────────────────────────────
def show_image(path, title=''):
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(10, 5))
        plt.imshow(img)
        plt.axis('off')
        if title:
            plt.title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f"File not found: {path}")

if bias_method == 'psm':
    show_image(os.path.join(config.RESULTS_DIR, 'psm_propensity_overlap_before.png'),
               'PSM — Propensity Score Overlap (before matching)')
    show_image(os.path.join(config.RESULTS_DIR, 'psm_propensity_overlap_after.png'),
               'PSM — Propensity Score Overlap (after matching)')
    for arm_id in sorted(k for k in config.TREATMENT_COMPONENTS if k != 0):
        show_image(os.path.join(config.RESULTS_DIR, f'psm_love_plot_arm{arm_id}.png'),
                   f'PSM Love Plot — Arm {arm_id} ({config.TREATMENTS[arm_id]})')
elif bias_method == 'iptw':
    show_image(os.path.join(config.RESULTS_DIR, 'iptw_weight_distribution.png'),
               'IPTW — Weight Distributions')
    for arm_id in sorted(k for k in config.TREATMENT_COMPONENTS if k != 0):
        show_image(os.path.join(config.RESULTS_DIR, f'iptw_love_plot_arm{arm_id}.png'),
                   f'IPTW Love Plot — Arm {arm_id} ({config.TREATMENTS[arm_id]})')

---
## Step 3 — Model 1: X-Learner Uplift

> ⏱️ **Expected runtime**: 2–8 minutes

In [ ]:
t3 = time.time()
print("Training X-Learner (3 offer arms vs. control) ...")

xlearner_model, auuc_df = train_xlearner(
    X             = X_for_xlearner,
    y             = y_for_xlearner,
    treatment     = t_for_xlearner,
    sample_weight = sample_weight_xl,
    save_results_dir = config.RESULTS_DIR,
)

# Predict CATEs on observed features (baseline scenario)
cates = xlearner_model.predict_all_cates(X_step2)
cates_path = os.path.join(config.RESULTS_DIR, 'cate_predictions.csv')
cates.to_csv(cates_path, index=False)

print(f"\n✅ X-Learner done  [{time.time()-t3:.1f}s]")
print(f"CATE columns: {list(cates.columns)}")
print(f"Saved to: {cates_path}")

In [ ]:
# ── AUUC table ────────────────────────────────────────────────────────────────
print("AUUC metrics by arm:")
display(auuc_df)

# ── CATE summary ──────────────────────────────────────────────────────────────
print("\nCATEs by arm:")
display(cates.describe().round(2))

In [ ]:
# ── Show uplift plots ─────────────────────────────────────────────────────────
show_image(os.path.join(config.RESULTS_DIR, 'uplift_curves.png'), 'Uplift Curves')

In [ ]:
# ── CATE distributions ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(cates.columns), figsize=(5 * len(cates.columns), 4))
if len(cates.columns) == 1:
    axes = [axes]

for ax, col in zip(axes, cates.columns):
    ax.hist(cates[col], bins=60, edgecolor='w', linewidth=0.3)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.2, label='No effect')
    ax.set_title(col.replace('cate_', '').replace('_', ' ').title())
    ax.set_xlabel('CATE ($)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('CATE Distributions by Arm', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 4 — Model 2: Attrition / Retention Prediction

In [ ]:
t4 = time.time()
print("Training attrition model ...")

attrition_model = train_attrition_model(
    X=X_step2, y=y_attrition, treatment=treatment,
    save_results_dir=config.RESULTS_DIR,
)

retention_proba = attrition_model.predict_proba(X_step2, treatment=treatment)
retention_df = pd.DataFrame({
    'retention_probability': retention_proba,
    'predicted_on_book':    (retention_proba >= 0.5).astype(int),
})
retention_path = os.path.join(config.RESULTS_DIR, 'retention_predictions.csv')
retention_df.to_csv(retention_path, index=False)

print(f"\n✅ Attrition model done  [{time.time()-t4:.1f}s]")
print(f"Retention predictions saved to: {retention_path}")
print(f"\nPredicted retention stats:")
display(retention_df.describe().round(3))

In [ ]:
show_image(os.path.join(config.RESULTS_DIR, 'attrition_roc_curve.png'), 'Attrition Model — ROC Curve')
show_image(os.path.join(config.RESULTS_DIR, 'attrition_pr_curve.png'), 'Attrition Model — Precision-Recall Curve')
show_image(os.path.join(config.RESULTS_DIR, 'attrition_feature_importance.png'), 'Attrition Model — Feature Importance')

---
## Step 4b — Combined Insights

In [ ]:
insights = pd.DataFrame({
    'treatment':                 treatment.values,
    'treatment_name':            df['treatment_name'].values,
    'offer':                     offers_col.values,
    'stipulation':               stip_col.values,
    'remail':                    remail_col.values,
    'opening_balance_actual':    y_balance.values,
    'retention_actual':          y_attrition.values,
    'retention_predicted_proba': retention_proba,
}, index=X_step2.index)

insights = pd.concat([insights, cates], axis=1)

insights_path = os.path.join(config.RESULTS_DIR, 'combined_insights.csv')
insights.to_csv(insights_path, index=False)
print(f"Combined insights saved to: {insights_path}")

# ── Campaign summary by offer arm ─────────────────────────────────────────────
print("\nCampaign Performance Summary (by offer amount):")
summary = insights.groupby('offer').agg(
    n_prospects               = ('opening_balance_actual', 'count'),
    mean_opening_balance      = ('opening_balance_actual', 'mean'),
    std_opening_balance       = ('opening_balance_actual', 'std'),
    mean_retention_actual     = ('retention_actual', 'mean'),
    mean_retention_predicted  = ('retention_predicted_proba', 'mean'),
).round(3)
display(summary)

# ── Campaign summary by stipulation × remail ──────────────────────────────────
print("\nCampaign Summary (stipulation × remail — treated only):")
summary2 = insights[insights['offer'] > 0].groupby(['stipulation', 'remail']).agg(
    n_prospects          = ('opening_balance_actual', 'count'),
    mean_opening_balance = ('opening_balance_actual', 'mean'),
    mean_retention       = ('retention_actual', 'mean'),
).round(3)
display(summary2)

---
## Step 5 — Model 3: Net Value Optimisation (Baseline)

In [ ]:
t5 = time.time()
print("Running net-value optimisation ...")

optimizer, net_value_results, strategy_comparison, qini_data, auuc_metrics = \
    optimize_offers(
        combined_insights_df = insights,
        save_results_dir     = config.RESULTS_DIR,
    )

print(f"\n✅ Net-value optimisation done  [{time.time()-t5:.1f}s]")
print("\nStrategy Comparison:")
display(strategy_comparison)

In [ ]:
show_image(os.path.join(config.RESULTS_DIR, 'optimal_offer_distribution.png'), 'Optimal Offer Distribution')
show_image(os.path.join(config.RESULTS_DIR, 'personalized_qini_curve.png'), 'Personalised Qini Curve')
show_image(os.path.join(config.RESULTS_DIR, 'personalized_cumulative_net_value.png'), 'Cumulative Net Value')
show_image(os.path.join(config.RESULTS_DIR, 'net_value_by_offer_boxplot.png'), 'Net Value by Offer')
show_image(os.path.join(config.RESULTS_DIR, 'strategy_comparison_bar.png'), 'Strategy Comparison')

---
## Step 5b — Decile Targeting Strategy

In [ ]:
t5b = time.time()
print("Evaluating decile targeting strategy (top-3 deciles) ...")

decile_strat_df, decile_breakdown_df = optimizer.evaluate_decile_targeting_strategy(
    df            = net_value_results,
    n_deciles     = 10,
    top_n_deciles = 3,
    save_dir      = config.RESULTS_DIR,
)

print(f"\n✅ Decile targeting done  [{time.time()-t5b:.1f}s]")
print("\nDecile Targeting Strategy Comparison:")
display(decile_strat_df)
print("\nDecile Breakdown:")
display(decile_breakdown_df.head(10))

In [ ]:
# Update comparison plots with personalised strategy overlay
xlearner_model.plot_auuc_comparison(
    auuc_df,
    net_value_auuc = auuc_metrics,
    save_path      = os.path.join(config.RESULTS_DIR, 'auuc_comparison.png'),
)
xlearner_model.plot_cumulative_gain(
    X_step2, y_balance, treatment,
    net_value_qini_data = qini_data,
    save_path           = os.path.join(config.RESULTS_DIR, 'cumulative_gain.png'),
)
show_image(os.path.join(config.RESULTS_DIR, 'auuc_comparison.png'), 'AUUC Comparison')
show_image(os.path.join(config.RESULTS_DIR, 'cumulative_gain.png'), 'Cumulative Gain')
show_image(os.path.join(config.RESULTS_DIR, 'decile_distribution.png'), 'Decile Distribution')
show_image(os.path.join(config.RESULTS_DIR, 'decile_vs_everyone_comparison.png'), 'Decile vs. Everyone')

---
## Step 6 — Scenario Analysis (remail × stipulation)

In [ ]:
t6 = time.time()
print("Running all remail × stipulation scenarios ...")
print("Scenarios:", list(config.OPTIMIZATION_SCENARIOS.keys()))

scen_summary, scenario_dfs = optimizer.run_all_scenarios(
    X_features     = X_step2,
    insights_df    = insights,
    xlearner_model = xlearner_model,
    save_dir       = config.RESULTS_DIR,
)

for scen_name, df_opt in scenario_dfs.items():
    scen_path = os.path.join(config.RESULTS_DIR, f'scenario_{scen_name}_results.csv')
    df_opt.to_csv(scen_path, index=False)

print(f"\n✅ Scenario analysis done  [{time.time()-t6:.1f}s]")
print("\nScenario Summary (sorted by total net value):")
display(scen_summary)

In [ ]:
show_image(os.path.join(config.RESULTS_DIR, 'scenario_comparison_bar.png'), 'Scenario Comparison')
show_image(os.path.join(config.RESULTS_DIR, 'scenario_offer_distributions.png'), 'Scenario Offer Distributions')

In [ ]:
# ── Best scenario ─────────────────────────────────────────────────────────────
best_row = scen_summary.iloc[0]
print("🏆 Best Scenario (highest total net value)")
print(f"   Scenario    : {best_row['scenario']}")
print(f"   remail      : {int(best_row['remail'])}")
print(f"   stipulation : {int(best_row['stipulation'])}")
print(f"   Total NV    : ${best_row['total_net_value']:,.2f}")
print(f"   Lift vs ctrl: ${best_row['lift_vs_control']:,.2f}")
print(f"   Extra cost/pp: ${best_row['extra_cost_per_prospect']:.2f}")

---
## Step 7 — Business Recommendations

Top-quartile CATE × retention segments per offer arm.

In [ ]:
print("Business Recommendations — top-quartile CATE × retention segments\n")
print("=" * 70)

recs = []
for arm_id in sorted(xlearner_model.models.keys()):
    cate_col = f'cate_treatment_{arm_id}'
    if cate_col not in insights.columns:
        continue

    high_cate      = insights[cate_col].quantile(0.75)
    high_retention = insights['retention_predicted_proba'].quantile(0.75)

    seg = (
        (insights[cate_col] >= high_cate) &
        (insights['retention_predicted_proba'] >= high_retention)
    )

    offer     = config.TREATMENT_COMPONENTS[arm_id]
    base_cost = config.OFFER_COST_RATE * offer

    recs.append({
        'Arm': arm_id,
        'Treatment': config.TREATMENTS[arm_id],
        'Segment size': seg.sum(),
        'Segment %': f"{100*seg.mean():.1f}%",
        'Avg CATE ($)': round(insights.loc[seg, cate_col].mean(), 2),
        'Avg P(retention)': f"{insights.loc[seg, 'retention_predicted_proba'].mean():.1%}",
        'Base cost ($)': base_cost,
        'With remail ($)': base_cost + config.REMAIL_COST,
        'With stip ($)': base_cost + config.STIPULATION_COST,
        'With both ($)': base_cost + config.REMAIL_COST + config.STIPULATION_COST,
    })

display(pd.DataFrame(recs).set_index('Arm'))

---
## Step 8 — Save Model Package

In [ ]:
t7 = time.time()
print(f"Saving model package to: {config.MODELS_DIR} ...")

save_pipeline(
    pruner          = pruner,
    boruta          = boruta,
    xlearner_model  = xlearner_model,
    attrition_model = attrition_model,
    save_dir        = config.MODELS_DIR,
    feature_names   = X_step2.columns.tolist(),
)

print(f"\n✅ Model package saved  [{time.time()-t7:.1f}s]")
print(f"\nTo score new prospects:")
print(f"  python src/scoring/score_new_data.py \\")
print(f"      --input  data/new_prospects.csv \\")
print(f"      --output results/scored_prospects.csv")

---
## 🏁 Pipeline Complete — Summary

In [ ]:
total_elapsed = time.time() - t_pipeline
print("=" * 70)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 70)
print(f"  ✓ Total time : {total_elapsed:.1f}s  ({total_elapsed/60:.1f} min)")
print(f"  📁 Results   : {config.RESULTS_DIR}")
print(f"  📦 Models    : {config.MODELS_DIR}")

summary_rows = [
    ('Data dimensions',           f"{X.shape[0]:,} samples × {X.shape[1]:,} features"),
    ('After Step 1 pruning',      f"{X_step1.shape[1]} features"),
    ('After Step 2 Boruta',       f"{X_step2.shape[1]} features"),
    ('Bias correction',           bias_method.upper()),
    ('X-Learner arms',            str(len(xlearner_model.models))),
    ('Best scenario',             best_row['scenario']),
    ('Best scenario total NV',    f"${best_row['total_net_value']:,.2f}"),
    ('Best scenario lift vs ctrl', f"${best_row['lift_vs_control']:,.2f}"),
]

print("\n" + "=" * 70)
df_summary = pd.DataFrame(summary_rows, columns=['Metric', 'Value']).set_index('Metric')
display(df_summary)